In [ ]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
RESULTS_DIR, EMOTIONS = config["paths"]["results_dir"], config["emotions"]
assert str(config.get("audio_feature_set", "")).startswith("wavlm") and str(config.get("video_backbone", "")).startswith("mtcnn"), "config.json describes the OLD features -- finish Steps 1-4 first."
ENABLED = os.environ.get("MELD_LOSS_SWEEP", "1").strip().lower() in ("1", "true", "yes")    
MIN_GAIN = float(os.environ.get("MELD_LOSS_MIN_GAIN", 0.01))
METRIC = os.environ.get("MELD_SELECT_METRIC", "macro_f1").strip().lower()      
assert METRIC in ("macro_f1", "accuracy"), "MELD_SELECT_METRIC must be 'macro_f1' or 'accuracy'"
SWEEP_PFX = "loss" if METRIC == "macro_f1" else "lossacc"                  
PER_SEED_FILE = "phase9_val_per_seed.csv" if METRIC == "macro_f1" else "phase9_val_per_seed_accuracy.csv"
FEATURE_KEYS = ("audio_feature_set", "audio_model", "video_backbone", "video_hsemotion_model", "text_feature_set", "audio_ft_feature_set")
FEATURE_DESC = {k: config.get(k) for k in FEATURE_KEYS if config.get(k) is not None}           
print(f"BASE_DIR={BASE_DIR} | results={RESULTS_DIR} | sweep enabled: {ENABLED} | min gain to replace the current loss: {MIN_GAIN}")

In [ ]:
VARIANTS = [   
    ("wce",           {}),
    ("wce_soft_p0.5", {"MELD_P9_LOSS": "wce_soft", "MELD_P9_WEIGHT_POWER": "0.5"}),
    ("focal_g1",      {"MELD_P9_LOSS": "focal", "MELD_P9_FOCAL_GAMMA": "1", "MELD_P9_WEIGHT_POWER": "0.5"}),
    ("focal_g2",      {"MELD_P9_LOSS": "focal", "MELD_P9_FOCAL_GAMMA": "2", "MELD_P9_WEIGHT_POWER": "0.5"}),
    ("logit_adj_t1",  {"MELD_P9_LOSS": "logit_adj", "MELD_P9_LA_TAU": "1.0"}),
]
SEEDS = os.environ.get("MELD_SWEEP_SEEDS", "42,43,44")      
LAMBDA = os.environ.get("MELD_SWEEP_LAMBDA", "2.0")         
if METRIC == "accuracy":     
    VARIANTS += [("ce_unweighted",  {"MELD_P9_LOSS": "wce_soft", "MELD_P9_WEIGHT_POWER": "0"}),
                 ("wce_soft_p0.25", {"MELD_P9_LOSS": "wce_soft", "MELD_P9_WEIGHT_POWER": "0.25"}),
                 ("focal_g2_p0",    {"MELD_P9_LOSS": "focal", "MELD_P9_FOCAL_GAMMA": "2", "MELD_P9_WEIGHT_POWER": "0"})]
FOCUS = ["fear", "disgust", "sadness"]

def variant_env(tag, overrides, seeds, lam, base_env):
    env = dict(base_env)
    env.update({"MELD_P9_LOSS": "wce", "MELD_P9_VERIFY": "0", "MELD_P9_SEEDS": seeds, "MELD_P9_LAMBDAS": lam, "MELD_P9_SWEEP_TAG": f"{SWEEP_PFX}_{tag}"})
    env.update(overrides)
    return env

def summarize(results_dir, tags, emotions, focus=FOCUS):
    """One row per (variant, model): validation macro-F1 / accuracy mean +- std over seeds, per-class F1 of the focus classes at each run's best epoch,
    and the robust selection score. Reads only the sweep outputs of Phase 9 (validation data)."""
    rows = []
    for tag in tags:
        d = os.path.join(results_dir, "sweeps", f"{SWEEP_PFX}_{tag}")
        per_seed, idx = os.path.join(d, "phase9_val_per_seed.csv"), os.path.join(d, "phase9_run_index.json")
        if not (os.path.exists(per_seed) and os.path.exists(idx)):
            continue
        ps, run_index = pd.read_csv(per_seed, index_col=0), json.load(open(idx))
        summ = pd.read_csv(os.path.join(d, "phase9_val_summary.csv")).set_index("name")
        for name in ps.columns:
            cls = {e: [] for e in emotions}
            for r in run_index:
                if r["name"] != name:
                    continue
                log = [json.loads(l) for l in open(os.path.join(r["run_dir"], "log.jsonl"))]
                for e, f in zip(emotions, log[r["best_epoch"]]["val_f1_per_class"]):
                    cls[e].append(f)
            row = {"variant": tag, "model": "baseline" if name == "baseline" else "proposed", "n_seeds": int(ps[name].count()),
                   "val_macro_f1": float(ps[name].mean()), "val_macro_f1_std": float(ps[name].std(ddof=1)) if ps[name].count() > 1 else float("nan"),
                   "val_accuracy": float(summ.loc[name, "best_val_accuracy_mean"]), "select_score": float(summ.loc[name, "best_select_score_mean"]) if "best_select_score_mean" in summ.columns else float("nan")}
            row.update({f"f1_{e}": float(np.mean(cls[e])) for e in focus if cls.get(e)})
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df, None
    piv = df.pivot(index="variant", columns="model", values="val_macro_f1")
    piv["mean_of_two"] = piv.mean(axis=1)
    piv["proposed_minus_baseline"] = piv["proposed"] - piv["baseline"] if {"proposed", "baseline"} <= set(piv.columns) else float("nan")
    return df, piv.sort_values("mean_of_two", ascending=False)
def per_seed_mean_of_two(results_dir, tags):
    out = {}
    for tag in tags:
        p = os.path.join(results_dir, "sweeps", f"{SWEEP_PFX}_{tag}", PER_SEED_FILE)
        if not os.path.exists(p):
            continue
        ps = pd.read_csv(p, index_col=0)
        prop = [c for c in ps.columns if c.startswith("proposed")]
        if "baseline" in ps.columns and len(prop) == 1:
            out[tag] = (ps["baseline"] + ps[prop[0]]) / 2.0
    return out

def decide_loss(per_seed, ref="wce", min_gain=0.01):
    if ref not in per_seed:
        raise RuntimeError(f"reference variant '{ref}' is missing from the sweep")
    means = {t: float(s.mean()) for t, s in per_seed.items()}
    best = max(means, key=means.get)
    if best == ref:
        return {"selected": ref, "reason": f"the current loss '{ref}' is already the best variant (mean {means[ref]:.4f})", "means": means}
    s_best, s_ref = per_seed[best], per_seed[ref].reindex(per_seed[best].index)
    gain, n_better = float((s_best - s_ref).mean()), int(((s_best - s_ref) > 0).sum())
    if gain >= min_gain and n_better == len(s_best):
        return {"selected": best, "reason": f"'{best}' gains {gain:+.4f} >= {min_gain} over '{ref}' and is better on all {n_better} sweep seeds", "means": means, "gain": gain}
    return {"selected": ref, "reason": f"best variant '{best}' gains only {gain:+.4f} (needs >= {min_gain} AND better on every seed; better on {n_better}/{len(s_best)}) -> keeping '{ref}'", "means": means, "gain": gain}
print("Helpers defined.")

In [ ]:
RUNNER_SRC = """
import sys, nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
src, dst, cwd = sys.argv[1:4]
nb = nbformat.read(src, as_version=4)
client = NotebookClient(nb, timeout=None, kernel_name="python3", resources={"metadata": {"path": cwd}})
ok = True
try:
    with client.setup_kernel():
        for i, cell in enumerate(nb.cells):
            if cell.cell_type != "code":
                continue
            client.execute_cell(cell, i)
            for o in cell.get("outputs", []):
                if o.get("output_type") == "stream":
                    t = o["text"]; print(t if len(t) < 3000 else t[:800] + "\\n...[truncated]...\\n" + t[-2000:], end="", flush=True)
except CellExecutionError as e:
    ok = False; print("CELL FAILED:", str(e)[-2500:], flush=True)
finally:
    nbformat.write(nb, dst)
sys.exit(0 if ok else 1)
"""
def find_phase9():
    for d in [os.environ.get("MELD_NOTEBOOK_DIR", ""), BASE_DIR, os.getcwd(), "/kaggle/working"]:
        if d and os.path.isfile(os.path.join(d, "Phase_09_Model_Training.ipynb")):
            return os.path.join(d, "Phase_09_Model_Training.ipynb")
    raise FileNotFoundError("Phase_09_Model_Training.ipynb not found (set MELD_NOTEBOOK_DIR). Use the STEP A version.")

nb9 = find_phase9()
assert "MELD_P9_SWEEP_TAG" in open(nb9).read(), "This Phase_09 notebook does not support the sweep (it lacks MELD_P9_SWEEP_TAG) -- use the STEP A version."
runner = os.path.join(BASE_DIR, "_run_notebook_sweep.py")
open(runner, "w").write(RUNNER_SRC)
os.makedirs(os.path.join(BASE_DIR, "executed_notebooks"), exist_ok=True)
for tag, ov in (VARIANTS if ENABLED else []):
    done = os.path.join(RESULTS_DIR, "sweeps", f"{SWEEP_PFX}_{tag}", "phase9_val_summary.csv")
    if os.path.exists(done):
        print(f"[{tag}] already complete -> skipped"); continue
    print(f"\n===== variant {tag}  ({ov or 'current loss'}) =====", flush=True)
    dst = os.path.join(BASE_DIR, "executed_notebooks", f"Phase_09_sweep_{tag}.ipynb")
    rc = subprocess.call([sys.executable, runner, nb9, dst, BASE_DIR], env=variant_env(tag, ov, SEEDS, LAMBDA, os.environ.copy()))
    if rc != 0:
        raise RuntimeError(f"variant {tag} failed (see {dst}). Fix the cause and re-run this notebook: finished variants/runs are skipped.")

In [ ]:
if ENABLED:
    df, piv = summarize(RESULTS_DIR, [t for t, _ in VARIANTS], EMOTIONS)
    pd.set_option("display.width", 220)
    cols = ["variant", "model", "n_seeds", "val_macro_f1", "val_macro_f1_std", "val_accuracy"] + [f"f1_{e}" for e in FOCUS] + ["select_score"]
    print(df[[c for c in cols if c in df.columns]].round(4).to_string(index=False))
    print("\nValidation macro-F1 per variant (rows sorted by the mean of the two models):")
    print(piv.round(4).to_string())
    df.to_csv(os.path.join(RESULTS_DIR, "loss_sweep_summary.csv"), index=False)
    best = piv.index[0]
    ref = piv.loc["wce", "mean_of_two"] if "wce" in piv.index else float("nan")
    if METRIC == "accuracy":
        print("\nMELD_SELECT_METRIC=accuracy: the table above shows macro-F1 / accuracy at the epochs chosen by validation ACCURACY; the DECISION below uses the mean per-seed validation accuracy of baseline and proposed.")
    print(f"\nSELECTION RULE (a decision you can change): highest mean validation macro-F1 of baseline and proposed, so the loss is not tuned in favour of either model.")
    print(f"Top variant: {best}  (mean {piv.loc[best, 'mean_of_two']:.4f}; current loss 'wce': {ref:.4f}; difference {piv.loc[best, 'mean_of_two'] - ref:+.4f}).")
    print("Per-seed std of validation macro-F1 is about 0.01: treat differences below that as NOT distinguishable and keep the current loss in that case.")
    print("Then run Phase 9 / Main with the chosen variant's env vars for BOTH models, e.g. MELD_P9_LOSS=focal MELD_P9_FOCAL_GAMMA=2 MELD_P9_WEIGHT_POWER=0.5")

In [ ]:
sel_path = os.path.join(RESULTS_DIR, "loss_sweep_selected.json")
ENV_OF = dict(VARIANTS)
if ENABLED:
    per_seed = per_seed_mean_of_two(RESULTS_DIR, [t for t, _ in VARIANTS])
    missing = [t for t, _ in VARIANTS if t not in per_seed]
    assert not missing, f"sweep variants without results: {missing}"
    dec = decide_loss(per_seed, ref="wce", min_gain=MIN_GAIN)
    seeds_used = [int(x) for x in per_seed["wce"].index]
else:
    dec, seeds_used = {"selected": "wce", "reason": "MELD_LOSS_SWEEP=0: sweep skipped, the current loss is kept", "means": {}}, []
selected = {"enabled": bool(ENABLED), "selected": dec["selected"], "env": ENV_OF[dec["selected"]], "reason": dec["reason"], "min_gain": MIN_GAIN,
            "mean_of_two_val_macro_f1": dec.get("means", {}), "metric_of_means": METRIC, "sweep_seeds": seeds_used, "sweep_lambda": float(LAMBDA), "feature_desc": FEATURE_DESC,
            "rule": f"highest mean validation {METRIC} of baseline and proposed; replaces the current loss only for gain >= min_gain on every seed", "select_metric": METRIC, "sweep_prefix": SWEEP_PFX,
            "note": "the loss is the same for baseline and proposed; test data was never used"}
tmp = sel_path + ".tmp"
json.dump(selected, open(tmp, "w"), indent=2); os.replace(tmp, sel_path)
print("DECISION:", dec["reason"])
print(f"Phase 9 will use: {selected['env'] or 'the current weighted cross-entropy (no overrides)'}  (explicit MELD_P9_LOSS still overrides)")
print("Phase 9a Loss sweep completed successfully.")